# DMOSpeech 2: Preliminary Test of the Switching Time on a 32-Interval Teacher Grid (EXP-017)

**Why.** The DMOSpeech 2 paper reports one teacher-guided setting: **t_switch = 0.25 with 14 teacher steps and 2 student steps** (Section 4.1). Its Algorithm 2 runs the teacher to the **last grid point at or below t_switch**, and 14 steps at 0.25 match a **32-interval** sway grid. Our earlier tests used the released sampler's default **16-interval** grid, where 0.25 becomes Step 8. This test repeats the fixed-switching comparison on the 32-interval grid and also measures the paper's own 14 + 2 setting.

| Output | Grid and stop | Teacher steps | Student steps | Forward passes |
|---|---|---:|---:|---:|
| `g16_4`: code default | 16 intervals, s = 0.07 (first point above) | 4 | 3 | 13 |
| `g16_8`: thesis reference | 16 intervals, s = 0.25 (first point above) | 8 | 3 | 21 |
| `s007` … `s020` | 32 intervals, s = 0.07 / 0.10 / 0.15 / 0.20 (last point ≤ s) | 7 / 9 / 11 / 13 | 3 | 19 / 23 / 27 / 31 |
| `s025`: reference on this grid | 32 intervals, s = 0.25 (last point ≤ s) | 14 | 3 | 33 |
| `paper`: the paper's 14 + 2, released handover | 32 intervals, s = 0.25 | 14 | 2 | 32 |
| `alg2`: the paper's Algorithm 2 as written | 32 intervals, s = 0.25; student from the teacher's state at 0.227, then 0.6173 | 14 | 2 | 30 |

Guidance, seeds and the student stay as in the thesis. `paper` restarts the student one level later so it takes two steps. `alg2` follows the paper's Algorithm 2: the student continues from the teacher's state without re-noising; its second time (0.6173, the student's last trained level) is assumed, because the paper does not state it. Failures follow Definition 3.1.

**Questions (descriptive, no pass/fail criteria).** Q1: on the 32-interval grid, do fixed early switching times fail mainly on noisy prompts and rarely on clean ones, as on the 16-interval grid (Appendix D)? Q2: does the cheapest safe switching time vary between inputs? Q3: how do the paper's 14 + 2 setting (both versions) and the code default compare in quality with the thesis reference?

**What it runs:** the EXP-012 Quick set (60 development speakers × clean, noisy15, Gaussian 18/20/22/25 dB = 360 cases), 9 scored outputs per case. About **3.5 hours for the full run on a T4, about 4 hours with setup** (estimate from the 10–13 s per case measured in EXP-013/014 for 2–3 outputs; Step 5 prints this run's seconds per case, and × 360 gives a firm estimate). **T4 only**: the runner stops on another GPU, because the student's random draws differ between GPU models. Development speakers only; the settings and questions are written to `preregistration.json` before any synthesis. Results are flushed to Google Drive after every case, so the run resumes after a disconnect.

---
### Instructions
1. Set the runtime to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
2. **Runtime > Run all.** Step 6 runs in the background; if the browser disconnects, reconnect and run **Step 6b**.
3. Check the Step 5 output before the full run finishes: the reproduction differences should be 0 (or about 1e-6), and the fresh Step-4 and Step-8 scores should equal the stored ones.
4. The last cell downloads `DMOSpeech2_Grid32_Results.zip`.

### Step 1: Verify GPU Allocation (must be a Tesla T4)

In [ ]:
!nvidia-smi
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('GPU:', name)
assert 'T4' in name, f"This test must run on a Tesla T4 (found {name}). Runtime > Change runtime type > T4 GPU."

### Step 2: Clone Repositories & Install Dependencies
`Shonget-DMOSpeech2` is checked out at `8a7868f`, the commit whose `core/` sampler produced the T4 grid. The thesis repository (branch `rhenelAudit`) supplies the runner, the split manifests and the stored labels; if it is private, add a Colab secret named `GITHUB_TOKEN`.

If Colab shows **"Restart session"** after the installs, click it, then continue from **Step 3**.

In [ ]:
import os, subprocess
DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}
!git log --oneline -1

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
THESIS_BRANCH = 'rhenelAudit'   # the branch that holds the EXP-017 runner
if not os.path.exists('/content/Final-Thesis-Setup'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git",
                    '/content/Final-Thesis-Setup'], check=True)
!git -C /content/Final-Thesis-Setup fetch -q origin {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup checkout -q {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup pull -q origin {THESIS_BRANCH}
!git -C /content/Final-Thesis-Setup log --oneline -1

# Same pinned dependencies as the earlier preliminary notebooks
!pip install -q "numpy==2.2.3" "pandas==2.2.3"
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

import numpy as np, pandas as pd, numba
print(f"Verified: NumPy {np.__version__} | Pandas {pd.__version__} | Numba {numba.__version__} [OK]")

runner_path = '/content/Final-Thesis-Setup/code/colab/run_grid32_prelim.py'
if not os.path.exists(runner_path):
    raise FileNotFoundError(f"{runner_path} is not on branch {THESIS_BRANCH}. Push it to GitHub before continuing.")

### Step 3: Download DMOSpeech 2 Checkpoints
`model_85000.pt` (4.0 GB, teacher + student) and `model_1500.pt` (270 MB, duration predictor) into `ckpts/`, checked against the SHA-256 hashes recorded for the earlier runs.

In [ ]:
%cd /content/Shonget-DMOSpeech2
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

import hashlib
for name, digest in {'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
                     'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'}.items():
    h = hashlib.sha256()
    with open(f'ckpts/{name}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    assert h.hexdigest() == digest, f"{name}: SHA-256 mismatch (delete it and re-run this step)"
    print(f"{name}: SHA-256 verified")

### Step 4: Download the Seed-TTS test-en Dataset

In [ ]:
import os
os.chdir('/content/Shonget-DMOSpeech2')

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'

# Check if audio is missing or only a Git LFS pointer
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print("Resolving Git LFS binaries for the Seed-TTS evaluation dataset...")
    if os.path.exists('data/seed-tts-eval/.git'):
        !cd data/seed-tts-eval && git lfs install && git lfs pull
    if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
        print("Cleaning leftover pointer files and downloading via huggingface_hub...")
        !rm -rf data/seed-tts-eval
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir="data/seed-tts-eval")
    print("Dataset setup complete!")
else:
    print("Seed-TTS audio binaries verified!")

!ls -lh data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav
!wc -l data/seed-tts-eval/en/meta.lst

### Step 5: Smoke Test + Reproduction Checks (2 development speakers, a few minutes)
Runs all 9 outputs for 2 non-pilot development speakers (clean and Gaussian 20 dB) into a **separate** `smoke/` folder. Two checks:
- the new grid-agnostic teacher loop, run on the released 16-interval grid, gives exactly the sampler's Step-4 and Step-8 endpoints (`max |diff|` 0);
- the fresh 16-interval Step-4 and Step-8 scores equal the stored T4 labels (they should on a T4).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'grid32_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_grid32_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

!python {RUNNER} smoke --out "{OUT}" --dmo_root {DMO}

import pandas as pd
sm = pd.read_csv(f'{OUT}/smoke/grid32.csv')
per_case = sm['case_seconds'].mean()
print(f"Smoke test OK: {len(sm)} cases, {per_case:.0f} s per case -> full run about {per_case * 360 / 3600:.1f} hours")
display(sm[['utt_id', 'condition', 'prompt_snr', 'case_seconds', 'd4_16', 'd4_stored',
            'wer_g16_4', 'wer_g16_4_stored', 'similarity_g16_4', 'similarity_g16_4_stored',
            'similarity_g16_8', 'similarity_g16_8_stored', 'similarity_s007', 'similarity_s025', 'similarity_paper', 'similarity_alg2']])

### Step 6: Full Run + Analysis
Writes `preregistration.json` (once), runs all 9 outputs for the 360 cases, then runs the analysis.

The job runs **in the background** and writes its full output to `MyDrive/dmospeech2_results/grid32_log.txt`; this cell prints one progress line per minute. If only the browser disconnects, the job keeps running; reconnect and run **Step 6b**.

**If the Colab machine itself is reset**, the results so far are safe in Google Drive (`MyDrive/dmospeech2_results/grid32_prelim/`). Re-run Steps 1–4, skip Step 5, then run this cell again: it continues from the last saved case.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'grid32_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_grid32_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

import subprocess, time, re
log_path = os.path.join(persist, 'grid32_log.txt')
cmd = f'python {RUNNER} run --out "{OUT}" --dmo_root {DMO}'
running = subprocess.run(['pgrep', '-f', 'run_grid32_prelim.py'], capture_output=True, text=True).stdout.split()
if running:
    pid = int(running[0])
    alive = lambda: os.path.exists(f'/proc/{pid}')
    print(f"A job is already running (pid {pid}); monitoring it instead of starting a new one.")
else:
    proc = subprocess.Popen(f'nohup {cmd} >> "{log_path}" 2>&1', shell=True)
    alive = lambda: proc.poll() is None
    print(f"Started (pid {proc.pid}). Full log: {log_path}")

last = ''
while alive():
    time.sleep(60)
    try:
        lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
        prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
        if prog and prog[-1] != last:
            last = prog[-1]
            print(time.strftime('%H:%M'), last)
    except FileNotFoundError:
        pass
print(f"Finished{'' if running else f' with exit code {proc.returncode}'}. Last lines of the log:")
for line in open(log_path, encoding='utf-8', errors='ignore').read().splitlines()[-15:]:
    print(line)

### Step 6b: Check Progress After a Reconnect
Shows whether the background job is still running and how many cases are saved. Does **not** start a second job.

In [ ]:
import os, re, subprocess
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'grid32_prelim')
log_path = os.path.join(persist, 'grid32_log.txt')
running = subprocess.run(['pgrep', '-f', 'run_grid32_prelim.py'], capture_output=True, text=True).stdout.split()
print("Job running:", "yes (pid " + ", ".join(running) + ")" if running else "no")
if os.path.exists(log_path):
    lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
    prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
    print("Latest progress:", prog[-1] if prog else "(none yet)")
    for line in lines[-5:]:
        print(line)
path = os.path.join(OUT, 'grid32.csv')
n = sum(1 for _ in open(path, encoding='utf-8')) - 1 if os.path.exists(path) else 0
print(f"grid32.csv: {n} of 360 cases saved")

### Step 7: View the Report
If the run stopped early, add `--allow_partial` for an interim look (the report is then marked PARTIAL).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'grid32_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_grid32_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

from IPython.display import Markdown, display
rep = os.path.join(OUT, 'GRID32_REPORT.md')
if not os.path.exists(rep) and os.path.exists(os.path.join(OUT, 'grid32.csv')):
    !python {RUNNER} analyze --out "{OUT}"
if os.path.exists(rep):
    display(Markdown(open(rep, encoding='utf-8').read()))
else:
    print(f"No report yet at {rep}. Run Step 6 first.")

### Step 8: Zip All Outputs and Download
Packs the CSV, preregistration, results JSON, report, smoke checks and the runners into one ZIP, also saved to Drive.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'grid32_prelim')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_grid32_prelim.py"'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

import zipfile, subprocess, datetime
zip_name = 'DMOSpeech2_Grid32_Results.zip'
with open(os.path.join(OUT, 'RUN_INFO.txt'), 'w') as f:
    f.write(f"dmospeech2_commit: {subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"thesis_commit: {subprocess.run(['git', '-C', THESIS, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.utcnow().isoformat()}Z\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_ in os.walk(OUT):
        dirs[:] = [d for d in dirs if d != 'listening']
        for fname in files_:
            if fname.startswith('tmp_') or fname.endswith('.wav'):
                continue
            full = os.path.join(root, fname)
            zf.write(full, os.path.relpath(full, OUT))
    for script in ('run_grid32_prelim.py', 'run_noise_guidance_prelim.py', 'run_endpoint_extrapolation_prelim.py', 'run_consistency_gate_prelim.py', 'run_followup_tests.py'):
        zf.write(f'{THESIS}/code/colab/{script}', f'code/{script}')
print(f"Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)")
!cp "{zip_name}" "{persist}/"
print(f"Also saved to Google Drive: {persist}/{zip_name}")
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f"Download manually from the Files panel: /content/Shonget-DMOSpeech2/{zip_name}")